# Module 04 — Offer Strength Simulator

**How to use**: Edit the `── YOUR PARAMETERS ──` cell below, then `Kernel → Restart & Run All`.

### What this produces
1. Zip market profile (hotness premium, price dispersion, DOM)
2. Single-point offer analysis with verdict
3. Bid sweep chart — win probability across a price range
4. Bid-for-probability table — "what do I need to bid for X% odds?"
5. Escalation clause Monte Carlo simulation
6. Full bid strategy report

### Model
Expected clearing price: `μ = fair_value + hotness_premium × dom_factor`  
Win probability: `P(win) = Φ((bid − μ) / σ)` where σ = empirical price std for the zip

In [13]:
# ── Setup — do not edit ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import joblib
from scipy import stats
import plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')

saved     = joblib.load('models/hedonic_model_filtered.joblib')
resid_std = saved['residual_std']

df = pd.read_csv('data/model_results_filtered.csv')
df['zip'] = df['zip'].astype(str).str.zfill(5)
cv_col = 'cv_predicted' if 'cv_predicted' in df.columns else 'predicted'
df['residual'] = df['PRICE'] - df[cv_col]

zip_stats = (
    df.groupby('zip')
    .agg(n=('PRICE','count'), med_price=('PRICE','median'),
         hotness_prem=('residual','median'), price_std=('residual','std'),
         city=('CITY', lambda x: x.mode().iloc[0] if len(x) > 0 else ''))
    .reset_index()
)
if 'DAYS_ON_MARKET' in df.columns:
    zip_stats = zip_stats.merge(
        df.groupby('zip')['DAYS_ON_MARKET'].median().rename('med_dom').reset_index(),
        on='zip', how='left')
else:
    zip_stats['med_dom'] = float('nan')
zip_stats['price_std']     = zip_stats['price_std'].fillna(resid_std)
zip_stats['hotness_score'] = (zip_stats['hotness_prem'].rank(pct=True) * 100).round(1)

def _zstats(z):
    row = zip_stats[zip_stats['zip'] == str(z).zfill(5)]
    if not row.empty and row.iloc[0]['n'] >= 5:
        r = row.iloc[0]
        return dict(zip=r['zip'], city=r['city'], n=int(r['n']),
                    hotness_prem=float(r['hotness_prem']), price_std=float(r['price_std']),
                    hotness_score=float(r['hotness_score']),
                    med_dom=float(r['med_dom']) if not pd.isna(r.get('med_dom', float('nan'))) else float('nan'))
    return dict(zip=z, city='Unknown', n=0,
                hotness_prem=float(zip_stats['hotness_prem'].median()),
                price_std=float(resid_std), hotness_score=50.0, med_dom=float('nan'))

def _dom_f(dom): return max(0.5, 1.0 - (dom - 30) / 300) if dom >= 30 else 1.0

def _mu(fv, z, dom, list_price=None):
    zs      = _zstats(z)
    list_p  = list_price if list_price is not None else fv
    gap_adj = (fv - list_p) * 0.35   # underpriced → +adj (more competition); overpriced → -adj
    mu      = fv + zs['hotness_prem'] * _dom_f(dom) + gap_adj
    return mu, zs, gap_adj

def _p_win(bid, fv, z, dom=0, list_price=None):
    mu, zs, gap_adj = _mu(fv, z, dom, list_price)
    return float(stats.norm.cdf(bid, mu, zs['price_std'])), zs, mu, gap_adj

def _bid_for_p(p, fv, z, dom=0, list_price=None):
    mu, zs, _ = _mu(fv, z, dom, list_price)
    return float(stats.norm.ppf(p, mu, zs['price_std']))

print(f'Ready. {len(df):,} sold homes | {df["zip"].nunique()} zip codes | σ = ${resid_std:,.0f}')

Ready. 5,078 sold homes | 101 zip codes | σ = $65,989


In [16]:
# ── YOUR PARAMETERS — edit this cell, then Kernel → Restart & Run All ────────

ZIP_CODE   = '19083'     # 5-digit zip code
FAIR_VALUE = 515_000     # Model fair-value estimate (from dashboard or 03_evaluation.ipynb)
LIST_PRICE = 559_900     # Seller's asking price (set equal to FAIR_VALUE if unknown)
YOUR_BID   = 515_000     # Your planned offer
DOM        = 45          # Current days on market
MAX_BUDGET = 600_000     # Your hard ceiling

# Escalation clause — set ESC_BASE = None to skip
ESC_BASE      = 500_000  # Opening escalation bid
ESC_INCREMENT = 10_000    # Beat any competing offer by this much
ESC_CAP       = 550_000  # Maximum you will pay

In [17]:
# ── Run analysis ─────────────────────────────────────────────────────────────
pw, zs, mu, gap_adj = _p_win(YOUR_BID, FAIR_VALUE, ZIP_CODE, DOM, LIST_PRICE)
sig        = zs['price_std']

# 1. Zip market profile
print(f'\n{"="*58}')
print(f'  ZIP {ZIP_CODE} — {zs["city"]}')
print(f'{"="*58}')
print(f'  Hotness premium (median above fair value):    ${zs["hotness_prem"]:>+10,.0f}')
print(f'  Price std dev (σ):                            ${sig:>10,.0f}')
print(f'  Hotness score:                               {zs["hotness_score"]:>10.0f}/100')
if not (isinstance(zs["med_dom"], float) and (zs["med_dom"] != zs["med_dom"])):
    print(f'  Median DOM:                                  {zs["med_dom"]:>10.0f} days')
print(f'  Comparable sales in dataset:                 {zs["n"]:>10,}')
list_adj_str = f'${gap_adj:>+10,.0f}  ({"underpriced" if gap_adj > 0 else "overpriced"})' if gap_adj != 0 else '     n/a (list = FV)'
print(f'  List price adjustment (0.35×gap):             {list_adj_str}')
if DOM >= 30:
    print(f'  DOM cooldown factor ({DOM} days):              {_dom_f(DOM):>10.2f}x')

# 2. Single-point offer analysis
print(f'\n{"-"*58}')
print(f'  OFFER ANALYSIS')
print(f'{"-"*58}')
print(f'  List price          : ${LIST_PRICE:>12,.0f}  (FV is {(FAIR_VALUE-LIST_PRICE)/LIST_PRICE*100:+.1f}% vs list)')
print(f'  Fair value (model)  : ${FAIR_VALUE:>12,.0f}')
print(f'  List price adj (0.35×gap): ${gap_adj:>+10,.0f}')
print(f'  Expected clear price: ${mu:>12,.0f}')
print(f'  Your bid            : ${YOUR_BID:>12,.0f}  ({(YOUR_BID-FAIR_VALUE)/FAIR_VALUE*100:+.1f}% vs fair value)')
print(f'  P(offer accepted)   : {pw*100:>11.1f}%')
if   pw >= 0.80: verdict = 'VERY STRONG  — likely the top bid'
elif pw >= 0.65: verdict = 'STRONG       — solid chance of acceptance'
elif pw >= 0.50: verdict = 'COMPETITIVE  — near the expected clearing price'
elif pw >= 0.35: verdict = 'MODERATE     — may lose to higher offers'
else:            verdict = 'WEAK         — well below likely clearing price'
print(f'  Verdict             : {verdict}')

# 3. Bid sweep chart
bids  = np.linspace(FAIR_VALUE * 0.82, min(FAIR_VALUE * 1.25, MAX_BUDGET * 1.06), 400)
probs = [float(stats.norm.cdf(b, mu, sig)) * 100 for b in bids]  # mu already includes gap_adj
fig = go.Figure()
fig.add_trace(go.Scatter(x=bids, y=probs, mode='lines', line=dict(color='#2980b9', width=3),
                         hovertemplate='Bid: $%{x:,.0f}<br>P(win): %{y:.1f}%<extra></extra>'))
for lo, hi, col, lbl in [(65,100,'rgba(39,174,96,0.09)','Strong (65%+)'),
                          (40,65, 'rgba(243,156,18,0.09)','Competitive (40–65%)'),
                          (0, 40, 'rgba(231,76,60,0.09)', 'Weak (<40%)')]:
    fig.add_hrect(y0=lo, y1=hi, fillcolor=col, line_width=0, annotation_text=lbl, annotation_position='right')
fig.add_vline(x=YOUR_BID,   line_dash='dash',  line_color='#e74c3c',
              annotation_text=f'Your bid ${YOUR_BID:,.0f} ({pw*100:.0f}%)', annotation_position='top right')
fig.add_vline(x=FAIR_VALUE, line_dash='dot',   line_color='#7f8c8d',
              annotation_text=f'Fair value ${FAIR_VALUE:,.0f}', annotation_position='top left')
fig.add_vline(x=mu,         line_dash='solid', line_color='#27ae60',
              annotation_text=f'Expected clear ${mu:,.0f}', annotation_position='bottom right')
if MAX_BUDGET <= bids[-1]:
    fig.add_vline(x=MAX_BUDGET, line_dash='dash', line_color='#8e44ad',
                  annotation_text=f'Max budget ${MAX_BUDGET:,.0f}', annotation_position='top right')
fig.update_layout(title=f'Win Probability vs Bid — ZIP {ZIP_CODE} ({zs["city"]})',
                  xaxis_title='Offer Price', yaxis_title='P(win) %',
                  xaxis_tickformat='$,.0f', yaxis=dict(range=[0,105], ticksuffix='%'),
                  height=420, plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()

# 4. Bid-for-probability table
print(f'\n{"-"*58}')
print(f'  BIDS NEEDED FOR TARGET WIN PROBABILITY')
print(f'{"-"*58}')
print(f'  {"Target":>8}  {"Required Bid":>14}  {"vs Fair Value":>13}  Notes')
print(f'  {"-"*54}')
for p in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    b    = _bid_for_p(p, FAIR_VALUE, ZIP_CODE, DOM, LIST_PRICE)
    dpct = (b - FAIR_VALUE) / FAIR_VALUE * 100
    note = '⚠ OVER BUDGET' if b > MAX_BUDGET else ('◄ YOUR BID' if abs(b - YOUR_BID) < 7000 else '')
    print(f'  {p*100:>7.0f}%  ${b:>13,.0f}  {dpct:>+12.1f}%  {note}')

# 5. Escalation clause simulation
if ESC_BASE is not None:
    rng = np.random.default_rng(42)
    hot = zs['hotness_score'] / 100
    n_probs = np.array([1-hot, hot*0.4, hot*0.4, hot*0.2]); n_probs /= n_probs.sum()
    wins, prices = [], []
    for _ in range(20_000):
        nc = rng.choice([0,1,2,3], p=n_probs)
        if nc == 0:
            wins.append(True); prices.append(ESC_BASE)
        else:
            best = rng.normal(mu, sig, nc).max()
            esc  = min(ESC_BASE + ESC_INCREMENT + max(0, best - ESC_BASE), ESC_CAP)
            won  = esc >= best
            wins.append(won)
            if won: prices.append(esc)
    wr  = float(np.mean(wins))
    avg = float(np.mean(prices)) if prices else float('nan')
    print(f'\n{"-"*58}')
    print(f'  ESCALATION CLAUSE (n=20,000 simulations)')
    print(f'{"-"*58}')
    print(f'  Base / Increment / Cap : ${ESC_BASE:,.0f} / ${ESC_INCREMENT:,.0f} / ${ESC_CAP:,.0f}')
    print(f'  Win rate               : {wr*100:.1f}%')
    print(f'  Avg final price if won : ${avg:,.0f}')
    print(f'  Probability-wtd cost   : ${wr*avg:,.0f}')

# 6. Bid strategy report
print(f'\n{"━"*58}')
print(f'  BID STRATEGY REPORT')
print(f'  ZIP {ZIP_CODE} ({zs["city"]})  |  List ${LIST_PRICE:,.0f}  |  FV ${FAIR_VALUE:,.0f}  |  DOM {DOM}')
print(f'{"━"*58}')
print(f'  {"Bid":>14}  {"vs Fair Val":>12}  {"P(win)":>8}  Tier')
print(f'  {"-"*54}')
for p_t in [0.30, 0.45, 0.55, 0.65, 0.75, 0.85]:
    b    = _bid_for_p(p_t, FAIR_VALUE, ZIP_CODE, DOM, LIST_PRICE)
    dpct = (b - FAIR_VALUE) / FAIR_VALUE * 100
    tier = '⚠ OVER BUDGET' if b > MAX_BUDGET else ('Strong' if p_t >= 0.75 else ('Competitive' if p_t >= 0.55 else 'Speculative'))
    you  = '  ◄ your bid' if abs(b - YOUR_BID) < 8000 else ''
    print(f'  ${b:>13,.0f}  {dpct:>+10.1f}%  {p_t*100:>7.0f}%  {tier}{you}')
print(f'  {"-"*54}')
print(f'  Max budget ${MAX_BUDGET:,.0f}  →  P(win) at cap = {_p_win(MAX_BUDGET, FAIR_VALUE, ZIP_CODE, DOM, LIST_PRICE)[0]*100:.1f}%')
print(f'{"━"*58}')


  ZIP 19083 — Havertown
  Hotness premium (median above fair value):    $    +4,129
  Price std dev (σ):                            $    77,610
  Hotness score:                                       58/100
  Comparable sales in dataset:                        198
  List price adjustment (0.35×gap):             $   -15,715  (overpriced)
  DOM cooldown factor (45 days):                    0.95x

----------------------------------------------------------
  OFFER ANALYSIS
----------------------------------------------------------
  List price          : $     559,900  (FV is -8.0% vs list)
  Fair value (model)  : $     515,000
  List price adj (0.35×gap): $   -15,715
  Expected clear price: $     503,208
  Your bid            : $     515,000  (+0.0% vs fair value)
  P(offer accepted)   :        56.0%
  Verdict             : COMPETITIVE  — near the expected clearing price



----------------------------------------------------------
  BIDS NEEDED FOR TARGET WIN PROBABILITY
----------------------------------------------------------
    Target    Required Bid  vs Fair Value  Notes
  ------------------------------------------------------
       30%  $      462,509         -10.2%  
       40%  $      483,545          -6.1%  
       50%  $      503,208          -2.3%  
       60%  $      522,870          +1.5%  
       70%  $      543,906          +5.6%  
       80%  $      568,526         +10.4%  
       90%  $      602,669         +17.0%  ⚠ OVER BUDGET

----------------------------------------------------------
  ESCALATION CLAUSE (n=20,000 simulations)
----------------------------------------------------------
  Base / Increment / Cap : $500,000 / $10,000 / $550,000
  Win rate               : 75.2%
  Avg final price if won : $509,789
  Probability-wtd cost   : $383,387

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  BID STRATEGY REPORT
  ZIP 1